# Day 2 — FEA pipeline validation on Kaggle CPU

**Purpose.** Stand up FEniCSx on Kaggle and validate the L-bracket FEA pipeline before the Day-3 parametric sweep. Everything that produces numerical results on the FEA side runs here; the local Windows repo is dev-only.

Sections:
1. Environment install (FEniCSx + gmsh via micromamba/conda-forge)
2. Smoke test: dolfinx canonical linear-elasticity tutorial
3. Ship our `src/fea` code into the notebook (inline, self-contained)
4. Mesh convergence study at the nominal midpoint sample
5. Analytical cross-check on an un-notched L
6. Load calibration on the worst-case sample (peak vm ~= 0.5 * sigma_y)
7. Emit `day2_results.json` + convergence plot for download

## 1. Install FEniCSx + gmsh

Kaggle Docker images don't ship FEniCSx, and there are no pip wheels. Reliable path: micromamba + conda-forge into an isolated env, then swap the notebook kernel to it with `sys.path` and ipykernel hook. Expected runtime: ~4-8 min on first run (cached thereafter).

In [ ]:
import os, subprocess, sys, time

MAMBA_ROOT = "/opt/conda-fenics"
ENV = f"{MAMBA_ROOT}/envs/fenicsx"
MAMBA_BIN = "/usr/local/bin/micromamba"

if not os.path.exists(MAMBA_BIN):
    print("Installing micromamba...")
    subprocess.check_call(
        "curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest "
        "| tar -xvj -C /usr/local bin/micromamba && "
        "chmod +x /usr/local/bin/micromamba",
        shell=True,
    )
    assert os.path.exists(MAMBA_BIN), f"expected {MAMBA_BIN} after install"
else:
    print("micromamba already present")

if not os.path.exists(ENV):
    print("Creating fenicsx env (this takes several minutes)...")
    t0 = time.time()
    # Pin fenics-dolfinx=0.9.* so the LinearProblem / read_from_msh / etc.
    # APIs match what src/fea/solver.py was written against. The 0.10 release
    # introduced breaking changes (e.g. LinearProblem now requires
    # petsc_options_prefix); the reference tutorial we archived in
    # raw/papers/dokken_fenicsx.md is also 0.9-series.
    subprocess.check_call(
        f"{MAMBA_BIN} create -y -p {ENV} -r {MAMBA_ROOT} -c conda-forge "
        "python=3.12 'fenics-dolfinx=0.9.*' mpich pyvista python-gmsh "
        "numpy scipy matplotlib",
        shell=True,
    )
    print(f"env ready in {time.time()-t0:.1f}s")
else:
    print("fenicsx env already created")

# Build the subprocess env that will run any cell needing FEniCSx. The key
# variable is LD_LIBRARY_PATH — the dynamic linker reads it at process start,
# so mutating os.environ in the running notebook kernel is insufficient.
# A subprocess of the env's own python, launched with env= set, resolves
# the shared libs cleanly.
ENV_BIN = f"{ENV}/bin"
ENV_PY = f"{ENV_BIN}/python"
FENICS_ENV = dict(os.environ)
FENICS_ENV["PATH"] = ENV_BIN + ":" + FENICS_ENV.get("PATH", "")
FENICS_ENV["LD_LIBRARY_PATH"] = f"{ENV}/lib:" + FENICS_ENV.get("LD_LIBRARY_PATH", "")

out = subprocess.check_output(
    [ENV_PY, "-c", "import dolfinx; print('dolfinx', dolfinx.__version__)"],
    env=FENICS_ENV,
)
print(out.decode())


## 2. Consolidated FEA validation (subprocess-driven)

Everything that needs dolfinx runs as a subprocess of the env's Python so that
`LD_LIBRARY_PATH` is picked up by the dynamic linker. The script emits a
single JSON bundle; the following cell plots and summarises.

In [ ]:
import base64, pathlib, subprocess, textwrap

OUT = pathlib.Path('/kaggle/working/day2')
OUT.mkdir(parents=True, exist_ok=True)

# Embedded src/fea payload (base64 of the assembled inline file).
_FEA_BLOB = '...'
(OUT / 'src_fea_inline.py').write_bytes(base64.b64decode(_FEA_BLOB))
print('wrote', OUT / 'src_fea_inline.py')



# Copy the inlined src/fea next to the runner so it can exec() it.

RUNNER = OUT / "run_all.py"
RUNNER.write_text(textwrap.dedent(r'''
    import json, pathlib, sys, numpy as np

    HERE = pathlib.Path(__file__).resolve().parent
    # Pull in the inlined src/fea modules — everything ends up in globals().
    exec((HERE / "src_fea_inline.py").read_text(), globals())

    # --- 1. dolfinx smoke test (canonical cantilever) --------------------
    from mpi4py import MPI
    import dolfinx
    from dolfinx import fem, mesh as dmesh
    from dolfinx.fem.petsc import LinearProblem
    from petsc4py import PETSc
    import ufl

    Lg, Hg = 1.0, 0.2
    m_sm = dmesh.create_rectangle(MPI.COMM_WORLD, [[0,0],[Lg,Hg]], [60,12],
                                  cell_type=dmesh.CellType.triangle)
    Vs = fem.functionspace(m_sm, ("Lagrange", 2, (2,)))
    left = dmesh.locate_entities_boundary(m_sm, 1, lambda x: np.isclose(x[0], 0.0))
    dofs = fem.locate_dofs_topological(Vs, 1, left)
    bc_sm = fem.dirichletbc(np.zeros(2, dtype=PETSc.ScalarType), dofs, Vs)
    Es, nus = 1.0e5, 0.3; mus = Es/(2*(1+nus)); lm = Es*nus/(1-nus*nus)
    def eps(u): return ufl.sym(ufl.grad(u))
    def sg(u): return lm*ufl.tr(eps(u))*ufl.Identity(2) + 2*mus*eps(u)
    u = ufl.TrialFunction(Vs); v = ufl.TestFunction(Vs)
    f = fem.Constant(m_sm, PETSc.ScalarType((0.0, -1.0)))
    uh = LinearProblem(ufl.inner(sg(u), eps(v))*ufl.dx,
                       ufl.inner(f, v)*ufl.dx, bcs=[bc_sm],
                       petsc_options={"ksp_type":"preonly","pc_type":"lu"}).solve()
    tip_u = uh.x.array.reshape(-1,2)[np.argmax(m_sm.geometry.x[:,0])]
    smoke = {"tip_u_x": float(tip_u[0]), "tip_u_y": float(tip_u[1])}
    print("smoke test tip u:", smoke)

    # --- 2. Mesh convergence on nominal midpoint -------------------------
    NOMINAL = dict(R=6.5, p=57.0, W=19.0)
    W_WORST = dict(R=3.0, p=42.0, W=14.0)
    PROBE_LOAD = 1.0

    levels = [
        dict(h_coarse=4.0, h_fine=1.5, refine_dist=6.0),
        dict(h_coarse=3.0, h_fine=0.8, refine_dist=6.0),
        dict(h_coarse=2.5, h_fine=0.4, refine_dist=7.0),
        dict(h_coarse=2.0, h_fine=0.2, refine_dist=8.0),
    ]
    params_nom = LBracketParams(**NOMINAL)
    conv = []
    for i, lvl in enumerate(levels):
        msh = HERE / f"nominal_L{i}.msh"
        write_msh(params_nom, msh,
                  h_coarse=lvl["h_coarse"], h_fine=lvl["h_fine"],
                  refine_dist=lvl["refine_dist"])
        r = solve_lbracket(msh, params_nom, PROBE_LOAD,
                           h_fine=lvl["h_fine"], h_coarse=lvl["h_coarse"],
                           return_fields=False)
        print(f"L{i} h_fine={lvl['h_fine']:.2f} dofs={r.n_dofs} peak_vm={r.peak_vm_mpa:.4f}")
        conv.append(dict(level=i, **lvl, peak_vm=float(r.peak_vm_mpa),
                         n_dofs=int(r.n_dofs), peak_xy=r.peak_location_xy))
    rel_two_finest = abs(conv[-1]["peak_vm"] - conv[-2]["peak_vm"]) / conv[-1]["peak_vm"]

    # --- 3. Analytical cross-check on un-notched L -----------------------
    import gmsh
    W_test = 19.0
    simple_geo = build_simplified_geo(W_mm=W_test, h_coarse=2.5, h_fine=0.3)
    geo_path = HERE / "simple.geo"; geo_path.write_text(simple_geo)
    msh_s = HERE / "simple.msh"
    gmsh.initialize(); gmsh.option.setNumber("General.Terminal", 0)
    gmsh.merge(str(geo_path)); gmsh.model.mesh.generate(2); gmsh.model.mesh.setOrder(2)
    gmsh.write(str(msh_s)); gmsh.finalize()
    r_simple = solve_lbracket(msh_s, LBracketParams(R=1.0, p=50.0, W=W_test),
                              PROBE_LOAD, h_fine=0.3, h_coarse=2.5, return_fields=True)
    x_probe = W_test + 15.0
    pred = bending_stress_at_section(x_mm=x_probe, W_mm=W_test, load_w_mpa=PROBE_LOAD)
    coords, vm = r_simple.coords, r_simple.vm_field
    mask = (np.abs(coords[:,0]-x_probe) < 0.6) & \
           ((np.abs(coords[:,1]) < 0.6) | (np.abs(coords[:,1]-W_test) < 0.6))
    fea_fibre = float(vm[mask].max()) if mask.any() else float("nan")
    cross_ratio = fea_fibre / pred.sigma_bending if pred.sigma_bending else float("nan")
    print(f"analytic={pred.sigma_bending:.3f}  fea_fibre={fea_fibre:.3f}  ratio={cross_ratio:.3f}")

    # --- 4. Load calibration on worst-case sample ------------------------
    converged = levels[-1]
    params_worst = LBracketParams(**W_WORST)
    msh_w = HERE / "worst.msh"
    write_msh(params_worst, msh_w,
              h_coarse=converged["h_coarse"], h_fine=converged["h_fine"],
              refine_dist=converged["refine_dist"])
    r_worst = solve_lbracket(msh_w, params_worst, 1.0,
                             h_fine=converged["h_fine"], h_coarse=converged["h_coarse"])
    target_peak = 0.5 * 205.0
    w_calibrated = target_peak / r_worst.peak_vm_mpa
    print(f"worst peak_vm@w=1: {r_worst.peak_vm_mpa:.3f}  w_calibrated={w_calibrated:.4f}")

    # --- 5. Emit bundle ---------------------------------------------------
    bundle = dict(
        smoke_test=smoke,
        convergence=conv,
        rel_two_finest=float(rel_two_finest),
        analytical_cross_check=dict(
            W=W_test, x_probe=float(x_probe), load=PROBE_LOAD,
            analytical_sigma=float(pred.sigma_bending),
            fea_fibre=fea_fibre, ratio=float(cross_ratio)),
        load_calibration=dict(
            worst_case=W_WORST, peak_at_w1=float(r_worst.peak_vm_mpa),
            calibrated_w_mpa=float(w_calibrated), target_peak_mpa=target_peak),
        levels=levels, nominal=NOMINAL,
    )
    (HERE / "day2_results.json").write_text(json.dumps(bundle, indent=2))
    print(json.dumps(bundle, indent=2))
''').lstrip())
print(f"wrote {RUNNER}")

# Run the validation as a subprocess of the env's Python.
import subprocess
proc = subprocess.run([ENV_PY, str(RUNNER)], env=FENICS_ENV,
                      capture_output=True, text=True)
print("--- stdout ---")
print(proc.stdout[-4000:])
print("--- stderr (tail) ---")
print(proc.stderr[-2000:])
assert proc.returncode == 0, f"runner exited {proc.returncode}"

## 3. Plot + summarize

In [ ]:
import json, pathlib, matplotlib.pyplot as plt

OUT = pathlib.Path("/kaggle/working/day2")
bundle = json.loads((OUT / "day2_results.json").read_text())

conv = bundle["convergence"]
fig, ax = plt.subplots(figsize=(6,4))
ax.plot([r["h_fine"] for r in conv], [r["peak_vm"] for r in conv], "o-")
ax.set_xlabel("h_fine at fillet [mm]"); ax.set_ylabel("peak vm [MPa]")
ax.set_title("Mesh convergence — nominal, w=1 MPa")
ax.invert_xaxis(); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(OUT / "convergence.png", dpi=140)
plt.show()

print(f"relative change between two finest: {bundle['rel_two_finest']*100:.2f}%")
print(f"analytical sigma = {bundle['analytical_cross_check']['analytical_sigma']:.3f} MPa")
print(f"FEA fibre stress = {bundle['analytical_cross_check']['fea_fibre']:.3f} MPa")
print(f"ratio (target ~1.0) = {bundle['analytical_cross_check']['ratio']:.3f}")
print(f"worst-case peak vm @ w=1 MPa = {bundle['load_calibration']['peak_at_w1']:.3f} MPa")
print(f"calibrated w = {bundle['load_calibration']['calibrated_w_mpa']:.4f} MPa "
      f"(targets {bundle['load_calibration']['target_peak_mpa']} MPa peak)")